# Qwen3-8B on a Colab T4

Runs on a Colab runtime (VS Code Colab extension, or colab.research.google.com). The kernel
lives on the Colab VM, **not** on this machine, so the repository is cloned there.

**The model is 4-bit (NF4, fp16 compute), not the reference Qwen3-8B.** A T4 has 15 GB;
the bf16 weights alone are ~16.4 GB. A quantized model is a different model: record
`QUANT` beside any number produced here, and do not compare activations from it with
full-precision ones (Proposal C, `research/experiments/drift_probe/`).

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## Clone and install

The repository is private. Running the cell asks for a fine-grained GitHub token with
read-only *Contents* access to `ghassenov/Tekmor`; in VS Code the prompt is the input box
at the top of the window. The token is passed to git through the environment only, so it
never reaches the notebook file, the command line, the remote URL, or a saved traceback.

Colab ships torch, transformers and accelerate, so only `bitsandbytes` is installed; Tekmor
itself has no runtime dependencies and is imported from `src/`. `bitsandbytes` is a notebook-only
dependency: it is not in `pyproject.toml`.

In [2]:
import base64
import getpass
import os
import subprocess
import sys

BRANCH = "research/colab-notebook"
REPO = "https://github.com/ghassenov/Tekmor.git"

if not os.path.isdir("/content/Tekmor"):
    # VS Code shows this prompt in the input box at the top of the window.
    token = getpass.getpass("GitHub token: ").strip()
    if not token:
        raise RuntimeError("empty token: rerun and paste it into the box at the top")
    # The token travels in the environment, never in argv: a failed clone prints argv
    # into the traceback, and the traceback into the saved notebook.
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ | {
        "GIT_TERMINAL_PROMPT": "0",
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.extraHeader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic}",
    }
    subprocess.run(
        ["git", "clone", "-q", "--branch", BRANCH, REPO, "/content/Tekmor"], env=env, check=True
    )
    del token, basic, env
%cd /content/Tekmor
!git log --oneline -n 1
# Tekmor has no runtime dependencies, so src/ on the path is the whole install. An
# editable pip install would not do: its .pth file is only read at interpreter start.
sys.path.insert(0, "/content/Tekmor/src")
!pip install -q bitsandbytes

/content/Tekmor
76c4607 (HEAD -> main, origin/main, origin/HEAD) Merge pull request #18 from ghassenov/research/drift-probe


## Load Qwen3-8B in 4-bit

About 6 GB of VRAM. The first load downloads ~16 GB from Hugging Face; a new Colab session
downloads it again.

In [4]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen3-8B"
QUANT = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 has no native bf16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=QUANT, device_map="auto")
print(transformers.__version__, torch.__version__, torch.cuda.get_device_name())
print(f"{torch.cuda.memory_allocated() / 2**30:.1f} GiB allocated")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

5.16.1 2.11.0+cu128 Tesla T4
6.0 GiB allocated


## Smoke test: one proposal through the adapter

`Qwen3Adapter.load()` returns early when a model is already set, so the quantized model is
handed to it rather than changing the adapter.

In [5]:
from tekmor.defense import AgentState
from tekmor.runtime.qwen import Qwen3Adapter

adapter = Qwen3Adapter(tools=("read_document", "send_email"), model=model, tokenizer=tokenizer)
state = AgentState(task="Read invoice INV-01 and summarise it.")
print(adapter.propose(state, observations=[]))

Action(tool='read_document', args={'id': 'INV-01'})


## Proposal B: Qwen3-8B as the judge, AgentDojo held out

`docs/decisions.md` records Proposal B as *built, measured on a proxy judge, not adopted*:
the one held-out number it needs, a judge that separates run on AgentDojo, does not exist.
This runs `evaluation/alignment.py` with Qwen3-8B as the judge, endorsement on (the
setting the auditor is for), on the matrix (dev set) and all four AgentDojo suites
(held out). The prompt and the 0.5 threshold are the frozen ones; nothing is fitted here.

It runs in a subprocess that loads its own 4-bit copy, so the smoke-test model is freed
first; the weights are already in the Hugging Face cache. `agentdojo` is pinned to the
version in `uv.lock`. **Every number is for NF4 Qwen3-8B on a T4**, which the manifest
records as `judge_quant` and `device`.

In [6]:
import gc

del adapter, model
gc.collect()
torch.cuda.empty_cache()
!pip install -q agentdojo==0.1.35

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.4/192.4 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.5/370.5 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.7/184.7 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.9/397.9 kB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 33.9 MB/s eta 0:00:00


In [7]:
!PYTHONPATH=src python -m evaluation.alignment \
    --judge Qwen/Qwen3-8B --dtype float16 --quant nf4 --endorse --dojo

usage: alignment.py [-h] [--scenarios SCENARIOS] [--results RESULTS]
                    [--judge JUDGE] [--threshold THRESHOLD] [--dtype DTYPE]
                    [--endorse] [--dojo] [--suites SUITES [SUITES ...]]
                    [--limit LIMIT]
alignment.py: error: unrecognized arguments: --quant nf4


### The same run with Phi-3-mini as the judge

On the matrix, Phi-3-mini was the one judge that separated honest from tampered calls.
On CPU it took about 20 s per call, so its held-out AgentDojo row was never run
(`docs/decisions.md`, Proposal B). Here it runs unquantized in fp16, which fits in about
7.6 GB. The earlier dev-set row was bf16 on CPU, so compare this run's matrix row with it
before reading the AgentDojo row.

In [ ]:
!PYTHONPATH=src python -m evaluation.alignment \
    --judge microsoft/Phi-3-mini-4k-instruct --dtype float16 --endorse --dojo

### Bring the results back

`evaluation/results/` lives on the Colab VM and is gitignored, and a new session loses it.
Printing the processed metrics and the manifest keeps them in this notebook's saved
output; copy them into `docs/decisions.md` from there. `runs.jsonl` stays on the VM.

In [8]:
import pathlib

for run in sorted(pathlib.Path("evaluation/results/raw").glob("*-alignment")):
    print("=" * 20, run.name)
    print((run / "manifest.json").read_text())
    print((pathlib.Path("evaluation/results/processed") / run.name / "alignment.json").read_text())

IndexError: list index out of range

## Proposal C: the drift probe on Qwen3-8B (NF4)

This follows the pre-registered *Amendment 2* in `research/experiments/drift_probe/README.md`.
It is the same recipe, data, seeds and gate, with only the model changed. The GPU is
freed before it starts, and WikiText-2 is downloaded into the Hugging Face cache, where
`probe.py` looks for it. Features are cached in `results/Qwen3-8B-nf4/`, so a crash after
extraction doesn't repeat the forward passes.

In [ ]:
from huggingface_hub import snapshot_download

gc.collect()
torch.cuda.empty_cache()
snapshot_download(
    "Salesforce/wikitext", repo_type="dataset", allow_patterns="wikitext-2-raw-v1/train-*"
)

In [ ]:
!PYTHONPATH=src:. python -m research.experiments.drift_probe.probe \
    --model Qwen/Qwen3-8B --quant nf4

In [ ]:
report = pathlib.Path("research/experiments/drift_probe/results/Qwen3-8B-nf4/report.json")
print(report.read_text())